### Ler etapa bronze

In [0]:
df_customers = spark.table(
    "workspace.bronze.customers"
)

### Analisar dados

In [0]:
display(df_customers)

In [0]:
df_customers.printSchema()

In [0]:
print(
    f"Quantidade de registros: {df_customers.count()}"
)

### Procurar nulos

In [0]:
from pyspark.sql.functions import *

In [0]:
display(
    df_customers.select(
        [
            count(
                when(
                    col(c).isNull(),
                    c
                )
            ).alias(c)
            for c in df_customers.columns
        ]
    )
)

### Procurar duplicidades

In [0]:
total = df_customers.count()

print(
    f"Total registros: {total}"
)

In [0]:
distintos = df_customers.distinct().count()

print(
    f"Registros distintos: {distintos}"
)

### Remover duplicidades

In [0]:
df_customers_silver = (
    df_customers
    .dropDuplicates()
)

### Criar metadado na Silver

In [0]:
from pyspark.sql.functions import current_timestamp, from_utc_timestamp, lit

In [0]:
df_customers_silver = (
    df_customers_silver
    .withColumn(
        "data_processamento_silver",
       from_utc_timestamp(
           current_timestamp(),
           "America/Sao_Paulo"
       )
    )
    .withColumn(
        "camada",
        lit("silver")
    )
)

### Verificar resultado

In [0]:
display(df_customers_silver)

### Gravar na Silver

In [0]:
(
    df_customers_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.customers"
    )
)

### Validação


In [0]:
%sql
select
*
from
workspace.silver.customers